In [7]:
from typing import Optional, List, Type
from pydantic import BaseModel, ValidationError
import pandas as pd
import geopandas as gpd
from shapely import wkb

In [8]:
# Define  Pydantic model
# Metro Fire Facilities
class MetroRecord(BaseModel):
    X: Optional[float]
    Y: Optional[float]
    gnaf_confidence: Optional[float]
    distance_to_gnaf: Optional[float]
    gnaf_lat: Optional[float]
    gnaf_long: Optional[float]

# Rural Fire Facilities
class RuralRecord(BaseModel):
    X: Optional[float]
    Y: Optional[float]
    gnaf_confidence: Optional[float]
    distance_to_gnaf: Optional[float]
    gnaf_lat: Optional[float]
    gnaf_long: Optional[float]

# Bushfire Prone Areas
class BushfireProneRecord(BaseModel):
    geom: Optional[str]
    bpa_areaha: Optional[float]

# Fire History
class FireHistoryRecord(BaseModel):
    geom: Optional[str]
    allcount: Optional[int]
    yrsfrburn: Optional[int]
    firecount: Optional[int]
    burncount: Optional[int]

# Fire Management Zone
class FireManageZoneRecord(BaseModel):
    geom: Optional[str]
    zonetype: Optional[float]

# Renewable Projects
class RenewableProjectRecord(BaseModel):
    Region: Optional[str]
    X: float
    Y: float

def read_select(path: str, model_class: Type[BaseModel], usecols=None, filters=None, xy_cols=None, wkb_col=None, crs="EPSG:4326", encoding=None):
    encodings_to_try = ['utf-8', 'latin-1', 'iso-8859-1', 'cp1252']
    if encoding:
        encodings_to_try = [encoding] + encodings_to_try

    df = None
    for enc in encodings_to_try:
        try:
            df = pd.read_csv(path, usecols=usecols + (list(filters.keys()) if filters else []), encoding=encoding)
            break
        except UnicodeDecodeError:
            continue

    if df is None:
        raise ValueError(f"Cannot read file: {path}")

    # Apply filters (on categorical fields)
    if filters:
        for col, val in filters.items():
            df = df[df[col] == val]
        # Drop filter columns after use
        df = df.drop(columns=list(filters.keys()), errors="ignore")

    df = df.dropna(subset=usecols)

    # Convert to Pydantic model
    records: List[BaseModel] = []
    for row in df.to_dict(orient="records"):
        filtered_row = {k: v for k, v in row.items() if k in model_class.model_fields}
        try:
            records.append(model_class(**filtered_row))
        except Exception as e:
            print("Row failed validation:", filtered_row)
            print(e)

    # Convert to GeoDataFrame if needed
    if xy_cols:
        gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df[xy_cols[0]], df[xy_cols[1]]), crs=crs)
    elif wkb_col:
        df["geometry"] = df[wkb_col].apply(lambda x: wkb.loads(bytes.fromhex(x), hex=True))
        gdf = gpd.GeoDataFrame(df.drop(columns=[wkb_col]), geometry="geometry", crs=crs)
    else:
        gdf = df  # just return the DataFrame if no geometry

    # Convert to numpy array + keep columns
    X = df.drop(columns=["geometry"], errors="ignore").to_numpy(dtype=float, copy=True)
    columns = [c for c in df.columns if c != "geometry"]

    return records, gdf, (X, columns)

In [9]:
# Metro Fire Facilities
metro_records, metro_gdf, (metro_X, metro_cols) = read_select(
    "https://github.com/zhenweiw1/capstone/raw/refs/heads/main/Metro_Fire_Facilities.csv",
    model_class=MetroRecord,
    filters={"facility_operationalstatus": "OPERATIONAL", "facility_state": "VICTORIA"},
    usecols=["X","Y","gnaf_confidence","distance_to_gnaf","gnaf_lat","gnaf_long"],
    xy_cols=("X","Y")
)

In [10]:
rural_records, rural_gdf, (rural_X, rural_cols)= read_select(
    "https://github.com/zhenweiw1/capstone/raw/refs/heads/main/Rural_Country_Fire_Service_Facilities.csv",
    model_class=RuralRecord,
    filters={"facility_operationalstatus": "OPERATIONAL", "facility_state": "VICTORIA"},
    usecols=["X","Y","gnaf_confidence","distance_to_gnaf","gnaf_lat","gnaf_long"],
    xy_cols=("X","Y")
)

In [11]:
bushfire_records, bushfire_gdf,(bushfire_X, bushfire_cols) = read_select(
    "https://github.com/zhenweiw1/capstone/raw/refs/heads/main/vic_fire_bushfire_prone_area.csv",
    model_class=BushfireProneRecord,
    usecols=["geom","bpa_areaha"],
    wkb_col="geom"
)

ValueError: could not convert string to float: '...'

In [ ]:
fire_history_records, fire_history_gdf,(fire_history_X, fire_history_cols)  = read_select(
    "https://github.com/zhenweiw1/capstone/raw/refs/heads/main/vic_fire_history_latest.csv",
    model_class=FireHistoryRecord,
    usecols=["geom","allcount","yrsfrburn","firecount","burncount"],
    wkb_col="geom"
)

/tmp/ipython-input-3585406088.py:75: PydanticDeprecatedSince20: The `__fields__` attribute is deprecated, use `model_fields` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  filtered_row = {k: v for k, v in row.items() if k in model_class.__fields__}


In [ ]:
fire_manage_records, fire_manage_gdf,(fire_manage_X, fire_manage_cols) = read_select(
    "https://github.com/zhenweiw1/capstone/raw/refs/heads/main/vic_fire_manage_zone.csv",
    model_class=FireManageZoneRecord,
    usecols=["geom","zonetype"],
    wkb_col="geom"
)

/tmp/ipython-input-3585406088.py:75: PydanticDeprecatedSince20: The `__fields__` attribute is deprecated, use `model_fields` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  filtered_row = {k: v for k, v in row.items() if k in model_class.__fields__}


In [ ]:
renewable_records, renewable_gdf,(renewable_X, renewable_cols) = read_select(
    "https://github.com/zhenweiw1/capstone/raw/refs/heads/main/renewable_project_nem.csv",
    model_class=RenewableProjectRecord,
    filters={"Region":"VIC1"},
    usecols=["Region","X","Y"],
    xy_cols=("X","Y"),
    encoding='latin-1'
)

/tmp/ipython-input-2403141638.py:73: PydanticDeprecatedSince20: The `__fields__` attribute is deprecated, use `model_fields` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  filtered_row = {k: v for k, v in row.items() if k in model_class.__fields__}
